In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
import sys
import os

# Columns and constants

COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']
NUMERIC_COLS = list(set(COLUMNS[:-2]) - set(CATEGORICAL_COLS))

POISONED_MODEL_FILE = "kdd_detection_model_poisoned.keras"


In [2]:
def load_data(train_path, test_path):
    try:
        train_df = pd.read_csv(train_path, names=COLUMNS[:-1], usecols=range(42))
        test_df = pd.read_csv(test_path, names=COLUMNS[:-1], usecols=range(42))
    except FileNotFoundError as e:
        print(f"Error: {e}")
        sys.exit(1)
    return train_df, test_df


In [3]:
def poison_train_labels(train_df, poison_frac=0.10, random_state=42):
    poisoned = train_df.copy()
    rng = np.random.default_rng(random_state)

    attack_mask = poisoned['label'] != 'normal'
    attack_indices = poisoned[attack_mask].index.to_numpy()

    n_attack = len(attack_indices)
    n_poison = int(poison_frac * n_attack)

    if n_poison == 0:
        print("Warning: poison_frac too small, no samples poisoned.")
        return poisoned

    poisoned_indices = rng.choice(attack_indices, size=n_poison, replace=False)
    poisoned.loc[poisoned_indices, 'label'] = 'normal'

    print(f"Poisoning: flipped {n_poison} attack samples.")
    return poisoned


In [4]:
def preprocess(train_df, test_df):
    y_train = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1).values
    y_test = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1).values

    X_train = train_df.drop('label', axis=1)
    X_test = test_df.drop('label', axis=1)

    combined = pd.concat([X_train, X_test], axis=0)
    combined_enc = pd.get_dummies(combined, columns=CATEGORICAL_COLS, dtype=float)

    X_train_enc = combined_enc.iloc[:len(X_train)]
    X_test_enc = combined_enc.iloc[len(X_train):]

    scaler = StandardScaler()
    scaler.fit(X_train_enc[NUMERIC_COLS])
    X_train_enc[NUMERIC_COLS] = scaler.transform(X_train_enc[NUMERIC_COLS])
    X_test_enc[NUMERIC_COLS] = scaler.transform(X_test_enc[NUMERIC_COLS])

    return X_train_enc.values, X_test_enc.values, y_train, y_test


In [5]:
def build_model(input_shape):
    model = keras.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [6]:
def main():
    TRAIN_FILE = "data/KDDTrain+.txt"
    TEST_FILE  = "data/KDDTest+.txt"

    train_df, test_df = load_data(TRAIN_FILE, TEST_FILE)

    poisoned_train_df = poison_train_labels(train_df, poison_frac=0.10, random_state=42)

    X_train_pois, X_test, y_train_pois, y_test = preprocess(poisoned_train_df, test_df)

    model = build_model(X_train_pois.shape[1])
    model.fit(X_train_pois, y_train_pois, epochs=20, batch_size=64, validation_split=0.1)

    model.save(POISONED_MODEL_FILE)

    model.evaluate(X_test, y_test)

    preds = (model.predict(X_test[:20]) > 0.5).astype(int).ravel()
    preds

main()

Poisoning: flipped 5863 attack samples.


C:\Users\hibat_jze8h9z\AppData\Local\Temp\ipykernel_21676\320084827.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train_enc[NUMERIC_COLS] = scaler.transform(X_train_enc[NUMERIC_COLS])
C:\Users\hibat_jze8h9z\AppData\Local\Temp\ipykernel_21676\320084827.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test_enc[NUMERIC_COLS] = scaler.transform(X_test_enc[NUMERIC_COLS])


Epoch 1/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - accuracy: 0.9340 - loss: 0.2055 - val_accuracy: 0.9447 - val_loss: 0.1754
Epoch 2/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9443 - loss: 0.1781 - val_accuracy: 0.9460 - val_loss: 0.1717
Epoch 3/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9459 - loss: 0.1718 - val_accuracy: 0.9467 - val_loss: 0.1700
Epoch 4/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9466 - loss: 0.1697 - val_accuracy: 0.9456 - val_loss: 0.1689
Epoch 5/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9470 - loss: 0.1678 - val_accuracy: 0.9469 - val_loss: 0.1668
Epoch 6/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9474 - loss: 0.1667 - val_accuracy: 0.9471 - val_loss: 0.1673
Epoch 7/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9483 - loss: 0.1652 - val_accuracy: 0.9470 - val_loss: 0.1661
Epoch 8/20
1772/1772 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9479 - loss: 0